In [1]:
from dataclasses import asdict

import h5py
import numpy as np
from qiskit_ibm_runtime import EstimatorOptions

from h5_interface import adapt_dtype_for_h5

try:
    f.close()
except NameError:
    pass
f = h5py.File("mytestfile.hdf5", "w")


In [2]:
dset = f.create_dataset("mydataset", (100,), dtype='i')

In [3]:
dset.name

'/mydataset'

In [4]:
grp = f.create_group("subgroup")

In [5]:
dset2 = grp.create_dataset("another_dataset", (50,), dtype='f')
dset2.name

'/subgroup/another_dataset'

In [6]:
dset3 = f.create_dataset('subgroup2/dataset_three', (10,), dtype='i')
dset3.name

'/subgroup2/dataset_three'

In [7]:
f['subgroup2/dataset_three']

<HDF5 dataset "dataset_three": shape (10,), type "<i4">

In [8]:
for name in f:
    print(name)

mydataset
subgroup
subgroup2


In [9]:
"mydataset" in f

True

In [10]:
"dataset_three" in f

False

In [11]:
f.visit(print)

mydataset
subgroup
subgroup/another_dataset
subgroup2
subgroup2/dataset_three


In [12]:
dset.attrs['temperature'] = 99.5

In [13]:
f.visit(print)

mydataset
subgroup
subgroup/another_dataset
subgroup2
subgroup2/dataset_three


In [14]:
f["mydataset"]

<HDF5 dataset "mydataset": shape (100,), type "<i4">

In [15]:
f["newdataset"] = np.arange(10).reshape(2, 5)

In [16]:
def my_print(name, obj: h5py.Group):
    print(f"{name}:{obj}")
    for key in obj.attrs.keys():
        print(f"\t{key}: {obj.attrs[key]}")


f.visititems(my_print)

mydataset:<HDF5 dataset "mydataset": shape (100,), type "<i4">
	temperature: 99.5
newdataset:<HDF5 dataset "newdataset": shape (2, 5), type "<i8">
subgroup:<HDF5 group "/subgroup" (1 members)>
subgroup/another_dataset:<HDF5 dataset "another_dataset": shape (50,), type "<f4">
subgroup2:<HDF5 group "/subgroup2" (1 members)>
subgroup2/dataset_three:<HDF5 dataset "dataset_three": shape (10,), type "<i4">


In [17]:
f["newdataset"][*np.array([1, 3])]

np.int64(8)

In [18]:
str(dset3.name).split("/")[-1]

'dataset_three'

In [19]:
f["newdataset"][1]

array([5, 6, 7, 8, 9])

In [20]:
test2 = (1 + 1j) * np.ones(5)

In [21]:
f["newcomplexdataset"] = test2

In [22]:
f["newcomplexdataset"]

<HDF5 dataset "newcomplexdataset": shape (5,), type "<c16">

In [23]:
f.require_dataset("tester", (10, 1024), maxshape=(None, 1024), dtype=np.float64)

<HDF5 dataset "tester": shape (10, 1024), type "<f8">

In [24]:
f.require_dataset("testera", (10, 1024), maxshape=(None, 1024), dtype=np.int64)

<HDF5 dataset "testera": shape (10, 1024), type "<i8">

In [25]:
f.require_dataset("testerb", (10, 1024), maxshape=(None, 1024), dtype=np.complex128)

<HDF5 dataset "testerb": shape (10, 1024), type "<c16">

In [26]:
f.require_dataset("testerc", (10, 1024), maxshape=(None, 1024), dtype=np.bool)

<HDF5 dataset "testerc": shape (10, 1024), type "|b1">

In [27]:
f.require_dataset("testerd", (10, 1024), maxshape=(None, 1024), dtype=h5py.string_dtype())

<HDF5 dataset "testerd": shape (10, 1024), type "|O">

In [28]:
if not (h5py.check_string_dtype(f["testerd"].dtype) is None):
    f["testerd"][0, 0] = str(object())

In [29]:
np.array(f["testerd"])

array([[b'<object object at 0x7bfba1cdbad0>', b'', b'', ..., b'', b'',
        b''],
       [b'', b'', b'', ..., b'', b'', b''],
       [b'', b'', b'', ..., b'', b'', b''],
       ...,
       [b'', b'', b'', ..., b'', b'', b''],
       [b'', b'', b'', ..., b'', b'', b''],
       [b'', b'', b'', ..., b'', b'', b'']],
      shape=(10, 1024), dtype=object)

In [30]:
np.array(f["testerd"]).astype(str)

array([['<object object at 0x7bfba1cdbad0>', '', '', ..., '', '', ''],
       ['', '', '', ..., '', '', ''],
       ['', '', '', ..., '', '', ''],
       ...,
       ['', '', '', ..., '', '', ''],
       ['', '', '', ..., '', '', ''],
       ['', '', '', ..., '', '', '']], shape=(10, 1024), dtype='<U33')

In [31]:
f.require_dataset("testerd", (10, 1024), maxshape=(None, 1024), dtype=h5py.string_dtype())

<HDF5 dataset "testerd": shape (10, 1024), type "|O">

In [32]:
value = True
f["testerd"].attrs["a"] = adapt_dtype_for_h5(value)

In [33]:
f["testerd"].attrs["a"].dtype

dtype('bool')

In [34]:
import json


def custom_json(obj):
    if isinstance(obj, complex):
        return {'__complex__': True, 'real': obj.real, 'imag': obj.imag}

    raise TypeError(f'Cannot serialize object of {type(obj)}')


print(json.dumps({'6': 7, '4': 5 + 2j}, sort_keys=True, indent=4, default=custom_json))

{
    "4": {
        "__complex__": true,
        "imag": 2.0,
        "real": 5.0
    },
    "6": 7
}


In [37]:
def custom_json(obj):
    if isinstance(obj, h5py.Group):
        temp = dict(obj)
        temp["Attributes"] = dict(obj.attrs)
        return temp
    if isinstance(obj, h5py.Dataset):
        temp = {
            "Dataset": str(obj),
            "Attributes": dict(obj.attrs),
        }
        return temp
    return str(obj)


with h5py.File("../results/data/2025-08-34.hdf5", "r") as f:
    json_txt = json.dumps(f["2025-08-29_11-01-27"], sort_keys=True, indent=4, default=custom_json)
print(json_txt)

{
    "Attributes": {
        "Ansatz": "XXPlusYYRZAnsatz1",
        "HamiltonianType": "ZeroChargePenalty",
        "Last-Modified": "2025-08-29_11-01-27",
        "NumLayers": "2",
        "NumParameters": "30",
        "NumQubits": "8",
        "ProcessId": "42",
        "QiskitAerVersion": "0.17.1",
        "QiskitIBMRuntimeVersion": "0.41.1",
        "QiskitVersion": "2.1.2",
        "SimulatorType": "Statevector",
        "SolverName": "VQE",
        "SystemName": "FreeWilson2D",
        "n_x": "2",
        "n_y": "2",
        "r": 1.0
    },
    "CircuitParameters": {
        "Attributes": {
            "DIMENSION_LABELS": "['mass' 'IterationAxis' 'CircuitParameterAxis']",
            "NDataDims": "2"
        },
        "Dataset": "<HDF5 dataset \"CircuitParameters\": shape (25, 240, 30), type \"<f8\">"
    },
    "Data": {
        "Attributes": {},
        "evs": {
            "Attributes": {},
            "hamiltonian": {
                "Attributes": {
                    "DI

In [38]:
from typing import Any


def remove_keys(my_dict: dict[str, Any], keys: list[str]):
    for key in keys:
        if key in my_dict:
            del my_dict[key]
    for value in my_dict.values():
        if isinstance(value, dict):
            remove_keys(value, keys)


temp = json.loads(json_txt)
# remove_keys(temp, ['Attributes'])
remove_keys(temp, ['Dataset'])
print(json.dumps(temp, sort_keys=True, indent=4))

{
    "Attributes": {
        "Ansatz": "XXPlusYYRZAnsatz1",
        "HamiltonianType": "ZeroChargePenalty",
        "Last-Modified": "2025-08-29_11-01-27",
        "NumLayers": "2",
        "NumParameters": "30",
        "NumQubits": "8",
        "ProcessId": "42",
        "QiskitAerVersion": "0.17.1",
        "QiskitIBMRuntimeVersion": "0.41.1",
        "QiskitVersion": "2.1.2",
        "SimulatorType": "Statevector",
        "SolverName": "VQE",
        "SystemName": "FreeWilson2D",
        "n_x": "2",
        "n_y": "2",
        "r": 1.0
    },
    "CircuitParameters": {
        "Attributes": {
            "DIMENSION_LABELS": "['mass' 'IterationAxis' 'CircuitParameterAxis']",
            "NDataDims": "2"
        }
    },
    "Data": {
        "Attributes": {},
        "evs": {
            "Attributes": {},
            "hamiltonian": {
                "Attributes": {
                    "DIMENSION_LABELS": "['mass' 'IterationAxis']",
                    "NDataDims": "1"
            

In [37]:
from h5_interface import save_dict_as_attribute

my_options = EstimatorOptions()
my_options.environment.job_tags = [1,2,3,42]
temp =asdict(my_options)
print(temp)
print(type(temp["environment"]["job_tags"]))
save_dict_as_attribute(f["subgroup2"],
                       temp,
                       "test")

{'_VERSION': 2, 'max_execution_time': Unset, 'environment': {'log_level': 'WARNING', 'callback': None, 'job_tags': [1, 2, 3, 42], 'private': False}, 'simulator': {'noise_model': Unset, 'seed_simulator': Unset, 'coupling_map': Unset, 'basis_gates': Unset}, 'default_precision': Unset, 'default_shots': Unset, 'resilience_level': Unset, 'seed_estimator': Unset, 'dynamical_decoupling': {'enable': Unset, 'sequence_type': Unset, 'extra_slack_distribution': Unset, 'scheduling_method': Unset, 'skip_reset_qubits': Unset}, 'resilience': {'measure_mitigation': Unset, 'measure_noise_learning': {'num_randomizations': Unset, 'shots_per_randomization': Unset}, 'zne_mitigation': Unset, 'zne': {'amplifier': Unset, 'noise_factors': Unset, 'extrapolator': Unset, 'extrapolated_noise_factors': Unset}, 'pec_mitigation': Unset, 'pec': {'max_overhead': Unset, 'noise_gain': Unset}, 'layer_noise_learning': {'max_layers_to_learn': Unset, 'shots_per_randomization': Unset, 'num_randomizations': Unset, 'layer_pair_d

In [38]:
json_txt = json.dumps(f, sort_keys=True, indent=4, default=custom_json)
print(json_txt)
temp = json.loads(json_txt)
f["subgroup2"]["test"]["environment"].attrs["job_tags"]

{
    "Attributes": {},
    "mydataset": {
        "Attributes": {
            "temperature": 99.5
        },
        "Dataset": "<HDF5 dataset \"mydataset\": shape (100,), type \"<i4\">"
    },
    "newcomplexdataset": {
        "Attributes": {},
        "Dataset": "<HDF5 dataset \"newcomplexdataset\": shape (5,), type \"<c16\">"
    },
    "newdataset": {
        "Attributes": {},
        "Dataset": "<HDF5 dataset \"newdataset\": shape (2, 5), type \"<i8\">"
    },
    "subgroup": {
        "Attributes": {},
        "another_dataset": {
            "Attributes": {},
            "Dataset": "<HDF5 dataset \"another_dataset\": shape (50,), type \"<f4\">"
        }
    },
    "subgroup2": {
        "Attributes": {},
        "dataset_three": {
            "Attributes": {},
            "Dataset": "<HDF5 dataset \"dataset_three\": shape (10,), type \"<i4\">"
        },
        "test": {
            "Attributes": {
                "_VERSION": "2",
                "default_precision": "Unset"

array([ 1,  2,  3, 42])

In [39]:
json.loads(json.dumps("[1,2,3,4]", sort_keys=True, indent=4, default=custom_json))

'[1,2,3,4]'